# W32 · Isaac Sim / Isaac Lab 概览

> 阶段四第 2 周。从本周起进入 NVIDIA 的 GPU 仿真栈：Isaac Sim 是仿真器，
> Isaac Lab 是建在其上的机器人学习框架。本 notebook 建立全局认知，
> 跑通（或读懂）官方示例，理解 Isaac Lab 的 Manager-based 设计。

## 学习目标

1. 说清 **Isaac Sim 与 Isaac Lab 的分工**，以及它们与 Isaac Gym 的历史关系；
2. 描述 Isaac 技术栈的架构层次（Kit / USD / PhysX GPU / Tensor API），并解释「为什么快」；
3. 区分 Isaac Lab 的两种任务设计 workflow：**Manager-based** 与 **Direct**；
4. 说出一个 Manager-based 环境由哪些 Manager 组成（Scene/Actions/Observations/Rewards/…）；
5. 在有 GPU 的机器上按 `docs/phase4_isaac.md` 完成安装并跑通官方示例。

## ⚠️ 运行前提

**本 notebook 的所有代码 cell 均需要 NVIDIA GPU + Isaac Sim/Isaac Lab（见 `docs/phase4_isaac.md`），
当前一律保持未执行。** 命令行示例以 markdown 代码块给出，可直接复制到有 GPU 的环境运行；
代码模板供下周（W33 自定义任务）直接使用。每个示例都附「预期输出」描述，供执行后对照。

## 1. 历史与定位：三个名字别搞混

- **Isaac Gym**（2021，已停止维护）：纯 GPU 物理 + RL 环境的「轻量先驱」，
  论文 *Isaac Gym: High Performance GPU-Based Physics Simulation For Robot Learning*
  （[Makoviychuk et al., 2021](https://arxiv.org/abs/2108.10470)）证明了
  「物理仿真和神经网络都放同一块 GPU 上」可以让 PPO 训练提速 1–2 个数量级；
- **Isaac Sim**：完整的**机器人仿真器**——基于 Omniverse Kit 的应用框架，
  有 USD 场景、PhysX 5 物理、RTX 渲染、传感器模型（相机/激光雷达/IMU）、ROS2 桥；
- **Isaac Lab**：建在 Isaac Sim 之上的**机器人学习框架**，提供任务/环境抽象、
  MDP 组件管理器、域随机化工具，以及与 rsl_rl / rl_games / SB3 / skrl 的集成。

关系一句话：**Isaac Lab : Isaac Sim = Gymnasium : MuJoCo**——前者是 RL 接口层，后者是物理引擎。

```
┌───────────────────────────────────────────────┐
│  Isaac Lab（RL 框架：任务/MDP/训练集成）        │
├───────────────────────────────────────────────┤
│  Isaac Sim（仿真器：USD Stage + 传感器 + ROS2）│
├───────────────┬───────────────────────────────┤
│  PhysX 5 GPU  │  Omniverse Kit（扩展/渲染）    │
├───────────────┴───────────────────────────────┤
│  NVIDIA GPU（RTX 系列，CUDA）                  │
└───────────────────────────────────────────────┘
```

> 💡 类比：Omniverse Kit 像 VS Code 的内核（插件化应用框架），Isaac Sim 是装在 Kit 上的
> 「机器人仿真插件全家桶」，Isaac Lab 则是你写在上面的「训练脚本与库」。

## 2. 为什么快：数据不出 GPU

传统 CPU 仿真（MuJoCo/PyBullet）每步都要 **CPU 仿真 → 拷贝观测到 GPU → 网络前向 → 动作拷回 CPU**，
PCIe 带宽成为瓶颈。Isaac 的做法是**全链路 GPU 驻留**：

```
   ┌─────────────────────── GPU ───────────────────────┐
   │  PhysX 仿真 ──► Tensor 观测缓冲 ──► 策略网络      │
   │      ▲                                    │        │
   │      └────────── 动作缓冲 ◄───────────────┘        │
   │  （观测/动作/奖励全部是 torch.Tensor，device=cuda）  │
   └────────────────────────────────────────────────────┘
```

加上 **4096 个环境同时仿真**，单步 wall-clock 几乎不随环境数增长（直到显存/算力饱和），
于是 PPO 每小时能吃下数千万步交互。Rudin et al.（2022）
（[Learning to Walk in Minutes](https://arxiv.org/abs/2201.08117)）正是靠这套管线把
「四足学会走路」压缩到分钟级。W34 会定量分析吞吐与并行度，这里先建立直觉：
**并行的单位是「环境实例」，吞吐的单位是「仿真步/秒（FPS / SPS）」**。

### 硬件要求（官方建议）

| 项 | 最低 | 推荐 |
|----|------|------|
| GPU | RTX 2080（8 GB） | RTX 3080+ / A 系列（16 GB+） |
| 系统 | Ubuntu 22.04 / Windows 11 | Ubuntu 22.04/24.04 |
| 驱动 | 最新 NVIDIA 驱动（CUDA 12.x 兼容） | 同左 |
| 磁盘 | ~30 GB | 50 GB+（含资产缓存） |

安装步骤全部整理在 **`docs/phase4_isaac.md`**，本周练习 1 就是完成它。

## 3. Isaac Lab 的两种任务设计 Workflow

| | **Manager-based** | **Direct** |
|---|---|---|
| 思想 | MDP 拆成若干「管理器」，每个术语是配置项 | 一个类实现 `_pre_physics_step/_apply_action/_get_observations/_get_rewards/_get_dones` |
| 配置风格 | 声明式（`@configclass` + TermCfg 组合） | 命令式（直接写 PyTorch 代码） |
| 灵活性 | 术语可复用、可热插拔，适合团队协作 | 完全自由，适合复杂/非标准逻辑 |
| 入门难度 | 概念多，但改配置不用动逻辑 | 概念少，逻辑全在自己手里 |
| 典型任务 | Isaac-Cartpole-v0、Isaac-Velocity-* 四足系列 | Isaac-Quadcopter-Direct-v0、Isaac-Franka-Cabinet-Direct-v0 |

> 💡 类比：Manager-based 像「搭乐高」——官方给你标准积木（现成的观测/奖励函数），
> 你通过配置组合；Direct 像「捏橡皮泥」——想怎么捏怎么捏，但每个零件都要自己造。
> 下周 W33 两种都会动手。

## 4. Manager-based 设计：一个环境 = 一叠 Manager

Manager-based 环境（`ManagerBasedRLEnv`）把 MDP 的每个侧面交给一个 Manager，
你用 `@configclass` 声明每个 Manager 里有哪些 **Term**（术语）。核心成员：

| Manager | 配置类 | 职责 | 对应 Gym 概念 |
|---------|--------|------|---------------|
| Scene | `InteractiveSceneCfg` |  spawn 资产（机器人/地形/灯光/传感器） | 环境初始化 |
| Actions | `ActionsCfg` | 网络输出 → 关节命令（位置/速度/力矩） | `action_space` + 动作映射 |
| Observations | `ObservationsCfg` | 拼观测向量（可含噪声 corruption） | `observation_space` |
| Rewards | `RewardsCfg` | 各奖励项加权求和 | reward function |
| Terminations | `TerminationsCfg` | 判定 done（摔倒/出界/超时） | `terminated/truncated` |
| Events | `EventCfg` | reset 时随机化（DR 的主战场） | `reset()` 的随机性 |
| Commands | `CommandsCfg` | 采样目标（速度指令/目标点） | 任务目标 |
| Curriculum | `CurriculumCfg` | 按表现调难度 | 课程学习 |
| Recorder | `RecorderManagerCfg` | 录演示数据（模仿学习用） | 数据采集 |

关键设计：**每个 Term = `func` + `params` + `weight` + 触发时机**。
比如一个奖励项 `RewTerm(func=mdp.flat_orientation_l2, weight=-2.5, params={...})`——
函数是官方库里现成的，你只改权重和参数。这就是「改配置不动逻辑」的含义。

下面这个 cell 展示装好环境后「像 Gym 一样」交互 Isaac Lab 任务（需 GPU，未执行）：

In [ ]:
# 前提：已按 docs/phase4_isaac.md 装好 Isaac Sim + Isaac Lab，且有 NVIDIA GPU。
# 用 Gymnasium 风格接口与官方 CartPole 任务交互（64 个并行环境）。
import gymnasium as gym
import torch

# 注意：isaaclab_tasks 的 import 会触发任务注册，必须在 gym.make 之前
import isaaclab_tasks  # noqa: F401
from isaaclab_tasks.utils import parse_env_cfg

env_cfg = parse_env_cfg("Isaac-Cartpole-v0", num_envs=64)
env = gym.make("Isaac-Cartpole-v0", cfg=env_cfg)

print("观测空间:", env.observation_space)
print("动作空间:", env.action_space)
print("并行环境数:", env.unwrapped.num_envs)

obs, _ = env.reset()
print("观测 tensor:", type(obs), getattr(obs, "shape", None))

# 随机动作跑 100 步——注意观测/动作都是 GPU 上的 torch.Tensor
for step in range(100):
    action = torch.rand(env.action_space.shape, device=env.unwrapped.device) * 2 - 1
    obs, reward, terminated, truncated, info = env.step(action)
    if step % 25 == 0:
        print(f"step {step:>3}: reward_mean={reward.mean().item():.3f}, "
              f"done_rate={(terminated | truncated).float().mean().item():.2f}")

env.close()

# 预期输出（描述）：
# - 观测/动作空间为 Box(shape=(4,)) 与 Box(shape=(1,)) 左右（具体以版本为准）
# - obs 是 dict 或 tensor，位于 cuda 设备
# - 每 25 步打印一次平均奖励；随机策略下 cart 很快出界，done_rate 较高

## 5. 官方示例命令（复制到有 GPU 的环境）

```bash
# ① 列出全部内置任务（Manager-based + Direct）
./isaaclab.sh -p scripts/environments/list_envs.py
# 预期输出：一张任务表，如 Isaac-Cartpole-v0、Isaac-Ant-v0、
#          Isaac-Velocity-Rough-Anymal-C-v0、Isaac-Lift-Cube-Franka-v0 …

# ② 训练蚂蚁走路（rsl_rl 框架，4096 环境，无渲染加速）
./isaaclab.sh -p scripts/reinforcement_learning/rsl_rl/train.py \
    --task=Isaac-Ant-v0 --num_envs=4096 --headless
# 预期输出：日志目录 logs/rsl_rl/ant/<日期>/，终端滚动打印
#   iteration / mean_reward / mean_episode_length / fps
#   在 RTX 30 系以上显卡，fps 通常达到 1e4~1e5 量级

# ③ 用训好的 checkpoint 回放（少量环境 + 渲染窗口观察行为）
./isaaclab.sh -p scripts/reinforcement_learning/rsl_rl/play.py \
    --task=Isaac-Ant-v0 --num_envs=32 --checkpoint logs/rsl_rl/ant/<日期>/model.pt

# ④ 四足 ANYmal 走崎岖地形（阶段二四足任务的 Isaac 版对照）
./isaaclab.sh -p scripts/reinforcement_learning/rsl_rl/train.py \
    --task=Isaac-Velocity-Rough-Anymal-C-v0 --headless
```

## 6. 支持的 RL 框架

`./isaaclab.sh --install` 默认装齐四个框架的集成（`scripts/reinforcement_learning/` 下各有目录）：

| 框架 | 特点 | 适用 |
|------|------|------|
| **rsl_rl** | ETH 出品，legged robot 事实标准，PPO 极简 | 四足/人形（阶段二的对口框架） |
| **rl_games** | 高性能 PPO/SAC，支持不对称 actor-critic | 通用 |
| **sb3** | 你在阶段一已经会用 | 快速迁移已有经验 |
| **skrl** | 模块化，多算法，文档友好 | 想试新算法 |

另有 **robomimic/robosuite** 集成用于模仿学习（Capstone 选题三会用到）。

## 7. 小结与常见坑

- Isaac Lab 的环境**不是普通 Gym env**：观测是 GPU tensor、4096 环境并行、reset 是「按 env_id 部分重置」；
- 首次运行 `isaacsim` 会拉取扩展缓存（可长达 10 分钟）并要求接受 EULA——不是卡死；
- 服务器无显示器时**务必加 `--headless`**；
- Isaac Sim 5.x 要求 **Python 3.11**（4.x 要求 3.10），因此它必须装在**独立环境**，
  与本项目的 `.venv`（3.10–3.12 通用依赖）分开——详见 `docs/phase4_isaac.md`。

---

## ✏️ 练习

1. **完成安装**（★，约 1–2 小时，需 GPU 机器）
   按 `docs/phase4_isaac.md` 装好 Isaac Sim + Isaac Lab，跑通验证脚本
   `scripts/tutorials/00_sim/create_empty.py` 和 `list_envs.py`。
   **交付物**：`list_envs.py` 输出中任意 10 个任务名的记录（`journal/isaac_envs.md`）。
2. **读懂一条训练日志**（★，约 20 分钟）
   运行 ② 的 Ant 训练至少 50 个 iteration，从日志中摘录：fps、mean_reward 随 iteration 的变化趋势，
   并回答：为什么前几个 iteration 的 fps 明显偏低？
   **交付物**：`journal/ant_train_log.md`（含 3 行以上日志原文 + 你的解释）。
3. **对比两种 workflow**（★★，约 40 分钟）
   在 Isaac Lab 源码中找到 `Isaac-Cartpole-v0`（Manager-based）与
   `Isaac-Quadcopter-Direct-v0`（Direct）的实现文件，各画出「文件 → 类 → 关键配置/方法」的结构草图，
   并回答：如果要把「旋翼推力存在 10% 随机增益」加进任务，两种 workflow 分别改哪里？
   **交付物**：`journal/workflow_compare.md`。
4. **架构问答**（★★，约 30 分钟，纯思考）
   有同学问：「既然 Isaac 这么快，能不能把我阶段一的 DroneHoverEnv 原封不动搬上来？」
   从「观测张量化、物理并行、reset 语义」三个角度回答为什么必须重写，
   以及重写时哪些部分（奖励函数的形状、超参先验）可以保留。
   **交付物**：`journal/why_rewrite.md`，200 字以上。

## 参考答案

<details>
<summary>练习 1：完成安装（检查要点）</summary>

- `isaacsim` 命令能启动（首次 10 分钟缓存属正常）；
- `create_empty.py` 弹出黑色视口（headless 服务器则日志无报错退出）；
- `list_envs.py` 打印任务表。任选 10 个示例：
  `Isaac-Cartpole-v0`、`Isaac-Cartpole-RGB-Camera-Direct-v0`、`Isaac-Ant-v0`、
  `Isaac-Humanoid-v0`、`Isaac-Velocity-Flat-Anymal-C-v0`、`Isaac-Velocity-Rough-Anymal-C-v0`、
  `Isaac-Lift-Cube-Franka-v0`、`Isaac-Reach-Franka-v0`、`Isaac-Quadcopter-Direct-v0`、
  `Isaac-Cart-Double-Pendulum-Direct-v0`。
- 常见失败：忘激活虚拟环境（`ModuleNotFoundError: No module named 'isaacsim'`）、
  驱动过旧（Vulkan 报错）、Python 版本不是 3.11。
</details>

<details>
<summary>练习 2：读懂训练日志（参考答案）</summary>

日志典型字段：`Learning iteration 3/300`、`Mean reward: -2.31`、
`Mean episode length: 48.2`、`fps: 42103`。

前几个 iteration fps 偏低的原因：① GPU 显存分配与 CUDA kernel 编译（首次调用的 JIT/缓存建立）；
② PhysX 初始化与 GPU buffer 分配；③ 日志统计窗口包含启动开销。
稳定后 fps 应趋于恒定。mean_reward 应随 iteration 上升（Ant 随机策略约 -2 ~ 0，
收敛后可达数百量级，具体数值以实际版本为准）。
</details>

<details>
<summary>练习 3：两种 workflow 对比（参考答案）</summary>

- **Cartpole（Manager-based）**：`source/isaaclab_tasks/isaaclab_tasks/manager_based/classic/cartpole/`
  下的 `cartpole_env_cfg.py`（Scene/Actions/Observations/Rewards/Terminations/EventCfg 组合）
  + `mdp/` 目录里的奖励函数。
  加推力随机增益 → 在 **EventCfg** 里注册一个 `mode="interval"` 或 `"reset"` 的 EventTerm，
  随机缩放 action 项的 scale（或写自定义 event 函数改机器人参数）——不动训练主逻辑。
- **Quadcopter（Direct）**：`source/isaaclab_tasks/isaaclab_tasks/direct/quadcopter/quadcopter_env.py`，
  一个 `QuadcopterEnv(DirectRLEnv)` 类集中实现所有钩子。
  加随机增益 → 直接在 `_apply_action()` 或 `_reset_idx()` 里写 PyTorch 代码采样增益系数。

结构草图略（自己画的才有意义）。核心结论：**Manager-based 把随机化收编进 Event Manager，
Direct 把随机化写在代码里**；前者可配置可复用，后者直观但不可热插拔。
</details>

<details>
<summary>练习 4：为什么必须重写（参考答案）</summary>

① **观测张量化**：Gym 版用 numpy 单环境逐步交互，Isaac 版要求观测/奖励/reset 全部是对
`(num_envs, dim)` 的 torch tensor 做向量化运算——标量 for 循环会毁掉并行性；
② **物理并行**：不能再自己积分 `v += a*dt`，动力学由 PhysX 在 GPU 上对 4096 个刚体同时求解，
你的职责从「写动力学」变成「配置 USD 资产 + 写张量化的奖励/终止函数」；
③ **reset 语义**：Gym 的 reset 是整环境归零，Isaac 是 `_reset_idx(env_ids)` 部分重置
（别的环境还在跑），时间管理（`episode_length_buf`）也由框架接管。

可保留的：奖励函数的**形状与量纲先验**（误差平方 + 速度惩罚 + 能耗惩罚的配方）、
终止条件的阈值、PPO 超参的搜索范围——这些知识跨框架通用。
</details>

---

## 延伸阅读

- [Isaac Lab 官方文档](https://isaac-sim.github.io/IsaacLab/)（安装、教程、API reference）
- [GitHub: isaac-sim/IsaacLab](https://github.com/isaac-sim/IsaacLab)（源码与 issue）
- [Isaac Sim 文档](https://docs.isaacsim.omniverse.nvidia.com/)
- 论文：[Isaac Gym (Makoviychuk et al., 2021)](https://arxiv.org/abs/2108.10470)、
  [Learning to Walk in Minutes (Rudin et al., 2022)](https://arxiv.org/abs/2201.08117)
- [rsl_rl](https://github.com/leggedrobotics/rsl_rl)、[skrl](https://github.com/Toni-SM/skrl)